# Explicabilidade com os microdados do ENEM 2023: LIME, SHAP e QII

Na aula anterior vimos que um modelo treinado só com o questionário socioeconômico **seleciona muito menos**
candidatos PPI e de escola pública, mesmo sem usar cor/raça como entrada. Hoje a pergunta é: **por quê?**
Quais variáveis levam o modelo a essas decisões, para uma pessoa e para os grupos?

**Objetivos da aula**

1. Treinar um modelo "caixa-preta" (gradient boosting) para prever alto desempenho no ENEM.
2. **LIME:** explicações *locais* por meio de um modelo linear substituto em torno de um candidato.
3. **SHAP:** valores de Shapley, explicações locais e globais e a **decomposição da diferença entre grupos**.
4. **QII (Quantitative Input Influence):** influência medida por **intervenção**, aplicada a quantidades de
   interesse como a decisão individual e a **disparidade entre grupos**.
5. Comparar os três métodos e discutir seus limites: instabilidade, correlação entre variáveis e
   "explicar não é justificar".

**Dados:** microdados do ENEM 2023 (INEP) — mesmo recorte da aula de métricas de justiça.

## 1. Configuração

In [ ]:
!pip install -q shap lime

In [ ]:
import os, zipfile, urllib.request, warnings, itertools, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score, accuracy_score

import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
SEED = 42
rng = np.random.default_rng(SEED)

SUPERFICIE, TINTA, TINTA2 = "#fcfcfb", "#0b0b0b", "#52514e"
mpl.rcParams.update({
    "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE,
    "axes.edgecolor": "#c9c8c2", "axes.labelcolor": TINTA2, "text.color": TINTA,
    "xtick.color": TINTA2, "ytick.color": TINTA2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "lines.linewidth": 2, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "legend.frameon": False,
})
AZUL, LARANJA, VERMELHO, CINZA = "#2a78d6", "#eb6834", "#e34948", "#a8a7a0"

## 2. Dados

Mesmo procedimento da aula anterior: download do INEP (ou caminho local/Drive), leitura apenas das colunas
necessárias e recorte para **concluintes de 2023, não treineiros, presentes nas quatro provas e com redação válida**.

In [ ]:
URL_INEP    = "https://download.inep.gov.br/microdados/microdados_enem_2023.zip"
CAMINHO_ZIP = os.environ.get("ENEM_ZIP", "microdados_enem_2023.zip")
CAMINHO_CSV = os.environ.get("ENEM_CSV", "MICRODADOS_ENEM_2023.csv")
N_LINHAS    = None   # ex.: 500_000 para uma amostra rápida

if not os.path.exists(CAMINHO_CSV):
    if not os.path.exists(CAMINHO_ZIP):
        print("Baixando microdados do INEP… (pode levar alguns minutos)")
        urllib.request.urlretrieve(URL_INEP, CAMINHO_ZIP)
    with zipfile.ZipFile(CAMINHO_ZIP) as z:
        membro = [n for n in z.namelist() if n.upper().endswith("MICRODADOS_ENEM_2023.CSV")][0]
        with z.open(membro) as src, open(CAMINHO_CSV, "wb") as dst:
            while chunk := src.read(1 << 24):
                dst.write(chunk)

QUESTOES = [f"Q{i:03d}" for i in range(1, 26)]
NOTAS    = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO"]
COLUNAS  = (["TP_FAIXA_ETARIA", "TP_SEXO", "TP_COR_RACA", "TP_ST_CONCLUSAO", "TP_ESCOLA",
             "IN_TREINEIRO", "SG_UF_PROVA", "TP_LINGUA",
             "TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT",
             "TP_STATUS_REDACAO"] + NOTAS + QUESTOES)
tipos = {c: "float32" for c in NOTAS}
tipos.update({c: "category" for c in QUESTOES + ["TP_SEXO", "SG_UF_PROVA"]})

bruto = pd.read_csv(CAMINHO_CSV, sep=";", encoding="latin-1", nrows=N_LINHAS,
                    usecols=lambda c: c in COLUNAS, dtype=tipos)

pres = bruto[["TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT"]].eq(1).all(axis=1)
df = bruto[(bruto["TP_ST_CONCLUSAO"] == 2) & (bruto["IN_TREINEIRO"] == 0) & pres &
           (bruto["TP_STATUS_REDACAO"] == 1) & bruto["TP_ESCOLA"].isin([2, 3])].copy()
del bruto

df["MEDIA"] = df[NOTAS].mean(axis=1)
df["raca"]  = df["TP_COR_RACA"].map({1: "Branca", 2: "Preta", 3: "Parda", 4: "Amarela", 5: "Indígena"})
df["ppi"]   = df["raca"].map({"Branca": "Branca/Amarela", "Amarela": "Branca/Amarela",
                              "Preta": "PPI", "Parda": "PPI", "Indígena": "PPI"})
df["escola"] = df["TP_ESCOLA"].map({2: "Pública", 3: "Privada"})
df = df.dropna(subset=["ppi", "MEDIA"])

CORTE = df["MEDIA"].quantile(0.75)
df["alto_desempenho"] = (df["MEDIA"] >= CORTE).astype(int)
print(f"{len(df):,} participantes | corte do top 25%: {CORTE:.1f}")

## 3. Variáveis legíveis

Explicações só ajudam se o aluno entender as variáveis. Traduzimos os códigos do questionário
(confira no `Dicionário_Microdados_Enem_2023.xlsx`, que vem dentro do zip do INEP).

As respostas são letras ordenadas (A = menor escolaridade, renda ou quantidade). Codificamos cada letra
como um inteiro (A=0, B=1, …), o que mantém **uma coluna por pergunta**: as explicações ficam no nível
da pergunta, e não de colunas one-hot.

In [ ]:
NOMES = {
    "Q001": "Escolaridade do pai", "Q002": "Escolaridade da mãe",
    "Q003": "Ocupação do pai", "Q004": "Ocupação da mãe", "Q005": "Pessoas na residência",
    "Q006": "Renda familiar", "Q007": "Empregado(a) doméstico(a)", "Q008": "Banheiros",
    "Q009": "Quartos", "Q010": "Carros", "Q011": "Motocicletas", "Q012": "Geladeira",
    "Q013": "Freezer", "Q014": "Máquina de lavar", "Q015": "Secadora", "Q016": "Micro-ondas",
    "Q017": "Lava-louça", "Q018": "Aspirador de pó", "Q019": "Televisão", "Q020": "DVD",
    "Q021": "TV por assinatura", "Q022": "Celular", "Q023": "Telefone fixo",
    "Q024": "Computador", "Q025": "Internet",
    "TP_ESCOLA": "Escola privada", "TP_FAIXA_ETARIA": "Faixa etária",
    "SG_UF_PROVA": "UF da prova", "TP_LINGUA": "Prova de espanhol",
}
ROTULOS = {  # rótulos das categorias para as perguntas mais usadas em sala
    "Escolaridade do pai": ["Nunca estudou", "Fund. I incompleto", "Fund. I completo", "Fund. II completo",
                            "Médio completo", "Superior completo", "Pós-graduação", "Não sei"],
    "Renda familiar": ["Nenhuma", "Até 1.320", "1.320–1.980", "1.980–2.640", "2.640–3.300", "3.300–3.960",
                       "3.960–5.280", "5.280–6.600", "6.600–7.920", "7.920–9.240", "9.240–10.560",
                       "10.560–11.880", "11.880–13.200", "13.200–15.840", "15.840–19.800",
                       "19.800–26.400", "Acima de 26.400"],
    "Computador": ["Não", "Sim, um", "Sim, dois", "Sim, três", "Sim, quatro ou mais"],
    "Internet": ["Não", "Sim"], "Escola privada": ["Não (pública)", "Sim"],
    "Prova de espanhol": ["Não (inglês)", "Sim"],
}
ROTULOS["Escolaridade da mãe"] = ROTULOS["Escolaridade do pai"]

def codificar(df):
    X = pd.DataFrame(index=df.index)
    for c in QUESTOES:
        s = df[c].astype(str)
        if c == "Q005":                        # número de pessoas (1..20) -> 0..19
            X[NOMES[c]] = pd.to_numeric(s, errors="coerce").fillna(1).clip(1, 20).astype(int) - 1
        else:                                  # letras A, B, C… -> 0, 1, 2…
            X[NOMES[c]] = s.str[0].map(lambda l: ord(l) - 65 if l.isalpha() else 0).astype(int)
    X[NOMES["TP_ESCOLA"]]       = (df["TP_ESCOLA"] == 3).astype(int)
    X[NOMES["TP_FAIXA_ETARIA"]] = df["TP_FAIXA_ETARIA"].fillna(1).astype(int) - 1
    X[NOMES["TP_LINGUA"]]       = df["TP_LINGUA"].fillna(0).astype(int)
    UFS = sorted(df["SG_UF_PROVA"].astype(str).unique())
    X[NOMES["SG_UF_PROVA"]]     = df["SG_UF_PROVA"].astype(str).map({u: i for i, u in enumerate(UFS)})
    return X, UFS

X_all, UFS = codificar(df)
ROTULOS["UF da prova"] = UFS
FEATURES = list(X_all.columns)
X_all.head()

## 4. O modelo "caixa-preta"

Um `HistGradientBoostingClassifier` prevê se o candidato fica no top 25% (`alto_desempenho`).
Como na aula anterior, **cor/raça e sexo não são entradas**. Diferente da regressão logística, aqui não há
coeficientes para ler: precisamos de métodos de explicação.

> A UF entra como código inteiro. Árvores conseguem separar estados mesmo assim, mas a ordem não tem significado.

In [ ]:
N_MODELO = 300_000
amostra = df.sample(min(N_MODELO, len(df)), random_state=SEED)
X = X_all.loc[amostra.index]
y = amostra["alto_desempenho"].values
G = amostra[["ppi", "escola", "raca"]].reset_index(drop=True)

X_tr, X_te, y_tr, y_te, G_tr, G_te = train_test_split(
    X.reset_index(drop=True), y, G, test_size=0.3, random_state=SEED, stratify=y)
X_tr, X_te, G_tr, G_te = [d.reset_index(drop=True) for d in (X_tr, X_te, G_tr, G_te)]

modelo = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=31,
                                        early_stopping=True, random_state=SEED)
modelo.fit(X_tr, y_tr)
p_te = modelo.predict_proba(X_te)[:, 1]
yhat_te = (p_te >= 0.5).astype(int)
print(f"AUC: {roc_auc_score(y_te, p_te):.3f} | Acurácia: {accuracy_score(y_te, yhat_te):.3f}")

recap = pd.DataFrame({"ppi": G_te["ppi"], "escola": G_te["escola"], "y": y_te, "yhat": yhat_te})
display(recap.groupby("ppi")[["y", "yhat"]].mean().rename(columns={"y": "taxa real", "yhat": "taxa de seleção"}))
display(recap.groupby("escola")[["y", "yhat"]].mean().rename(columns={"y": "taxa real", "yhat": "taxa de seleção"}))

### A candidata que vamos explicar

Escolhemos no conjunto de teste uma pessoa **PPI, de escola pública, que de fato ficou no top 25%, mas que o
modelo rejeitou** (um falso negativo). Ela é o tipo de caso que motivaria um pedido de explicação.

In [ ]:
cand = recap[(recap["ppi"] == "PPI") & (recap["escola"] == "Pública") & (recap["yhat"] == 0)]
cand_fn = cand[cand["y"] == 1]
pool = cand_fn if len(cand_fn) else cand
i_cand = pool.index[np.argmax(p_te[pool.index])]   # a mais próxima do limiar: caso "de fronteira"
x_cand = X_te.loc[[i_cand]]

def legivel(col, v):
    if col in ("Pessoas na residência", "Faixa etária"):
        return str(int(v) + 1)
    r = ROTULOS.get(col)
    return r[int(v)] if r is not None and int(v) < len(r) else chr(65 + int(v)) if v < 26 else str(v)

print(f"Probabilidade prevista: {p_te[i_cand]:.3f} | Ficou no top 25%? {'sim' if y_te[i_cand] else 'não'}")
pd.DataFrame({"resposta": [legivel(c, x_cand[c].iloc[0]) for c in FEATURES]}, index=FEATURES).T

## 5. LIME — Local Interpretable Model-agnostic Explanations

**Ideia (Ribeiro, Singh & Guestrin, 2016):** perto de um ponto $x$, até um modelo complexo se parece com um
modelo simples. O LIME:

1. gera **perturbações** $z$ de $x$ (aqui: troca algumas respostas por respostas de outros candidatos);
2. consulta o modelo caixa-preta $f(z)$;
3. pondera cada $z$ pela **proximidade** a $x$, $\pi_x(z) = \exp(-d(x,z)^2/\sigma^2)$;
4. ajusta um modelo linear ponderado $g$; os coeficientes de $g$ são a explicação.

$$\xi(x) = \arg\min_{g \in G} \; \mathcal{L}(f, g, \pi_x) + \Omega(g)$$

### 5.1 Um LIME "de brinquedo", feito à mão

In [ ]:
def lime_na_mao(x, f, X_ref, n=5000, sigma=0.25, seed=0):
    r = np.random.default_rng(seed)
    p = x.shape[1]
    manter = r.integers(0, 2, size=(n, p))                     # 1 = mantém a resposta de x
    doadores = X_ref.values[r.integers(0, len(X_ref), size=n)]  # respostas de outras pessoas
    Z = np.where(manter == 1, x.values, doadores)
    dist = 1 - manter.mean(axis=1)                              # fração de respostas trocadas
    peso = np.exp(-(dist ** 2) / sigma ** 2)
    g = Ridge(alpha=1.0).fit(manter, f(pd.DataFrame(Z, columns=x.columns)), sample_weight=peso)
    return pd.Series(g.coef_, index=x.columns).sort_values(key=abs, ascending=False)

f_proba = lambda Z: modelo.predict_proba(Z)[:, 1]
exp_mao = lime_na_mao(x_cand, f_proba, X_tr)
exp_mao.head(10).to_frame("efeito de manter a resposta da candidata")

Interpretação: um coeficiente **negativo** em "Renda familiar" significa que manter a renda *desta candidata*,
em vez de uma renda sorteada da população, **reduz** a probabilidade prevista.

### 5.2 Com a biblioteca `lime`

In [ ]:
nomes_categorias = {j: [legivel(c, v) for v in range(int(X_all[c].max()) + 1)]
                    for j, c in enumerate(FEATURES)}
explainer_lime = LimeTabularExplainer(
    X_tr.values, mode="classification", feature_names=FEATURES,
    categorical_features=list(range(len(FEATURES))), categorical_names=nomes_categorias,
    class_names=["fora do top 25%", "top 25%"], discretize_continuous=False, random_state=SEED)

def explicar_lime(seed=SEED, n_features=10):
    explainer_lime.random_state = np.random.RandomState(seed)
    return explainer_lime.explain_instance(
        x_cand.values[0], lambda Z: modelo.predict_proba(pd.DataFrame(Z, columns=FEATURES)),
        num_features=n_features, num_samples=5000)

exp = explicar_lime()
pesos = pd.Series(dict(exp.as_list())).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(pesos.index, pesos.values, color=[AZUL if v > 0 else VERMELHO for v in pesos.values], height=0.6)
ax.axvline(0, color=TINTA2, linewidth=1); ax.grid(axis="y", visible=False)
ax.set_xlabel("Peso no modelo linear local  (azul: aumenta P(top 25%) · vermelho: diminui)")
ax.set_title("LIME: explicação para a candidata"); plt.tight_layout(); plt.show()
print(f"R² do modelo local: {exp.score:.3f}  (quão bem a reta aproxima o modelo perto da candidata)")

### 5.3 Instabilidade

O LIME é **aleatório**: perturbações diferentes geram explicações diferentes. Rodamos 10 vezes para a mesma
candidata e comparamos os pesos.

In [ ]:
execucoes = pd.DataFrame([dict(explicar_lime(seed=s, n_features=8).as_list()) for s in range(10)])
estab = pd.DataFrame({"peso médio": execucoes.mean(), "desvio-padrão": execucoes.std(),
                      "apareceu no top 8 (de 10)": execucoes.notna().sum()})
estab.sort_values("apareceu no top 8 (de 10)", ascending=False)

💬 **Para discutir:** se a explicação muda a cada execução, ela pode ser usada para contestar uma decisão?
Que parâmetros (número de amostras, `kernel_width`, forma de perturbar) você mudaria?

## 6. SHAP — SHapley Additive exPlanations

**Ideia (Lundberg & Lee, 2017):** tratar as variáveis como "jogadores" de um jogo cooperativo cujo "prêmio"
é a predição. O valor de Shapley de cada variável $i$ é sua contribuição marginal média em todas as ordens
possíveis de entrada no jogo:

$$\phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!\,(|N|-|S|-1)!}{|N|!}\,\big[v(S \cup \{i\}) - v(S)\big]$$

onde $v(S)$ é a predição média quando só as variáveis de $S$ são conhecidas. Propriedade-chave
(**eficiência**): $\sum_i \phi_i = f(x) - \mathbb{E}[f(X)]$. A predição se decompõe exatamente.

### 6.1 Shapley exato, à mão, com 4 jogadores

Com 4 variáveis há só $2^4 = 16$ coalizões. As demais variáveis ficam "ausentes": seus valores vêm de uma
amostra de fundo (*background*).

In [ ]:
JOGADORES = ["Renda familiar", "Escola privada", "Escolaridade da mãe", "Computador"]
fundo = X_tr.sample(300, random_state=SEED)
logodds = lambda Z: modelo.decision_function(Z)   # SHAP para árvores trabalha em log-odds

def v(S):
    Z = fundo.copy()
    for c in S:
        Z[c] = x_cand[c].iloc[0]
    return logodds(Z).mean()

n = len(JOGADORES)
phi = {}
for i in JOGADORES:
    outros = [j for j in JOGADORES if j != i]
    phi[i] = sum(math.factorial(len(S)) * math.factorial(n - len(S) - 1) / math.factorial(n)
                 * (v(list(S) + [i]) - v(list(S)))
                 for k in range(n) for S in itertools.combinations(outros, k))
phi = pd.Series(phi)
display(phi.to_frame("φ (log-odds)"))
print(f"Soma dos φ = {phi.sum():.4f} | v(todos) − v(∅) = {v(JOGADORES) - v([]):.4f}  ← eficiência")

### 6.2 TreeSHAP: explicação local

Para árvores, o `shap` calcula os valores de Shapley **exatos** em tempo polinomial (TreeSHAP).

In [ ]:
explainer = shap.TreeExplainer(modelo)
X_exp = X_te.sample(min(5000, len(X_te)), random_state=SEED)
sv = explainer(X_exp)
sv_cand = explainer(x_cand)

shap.plots.waterfall(sv_cand[0], max_display=12)

O gráfico parte de $\mathbb{E}[f(X)]$ (a predição média, em log-odds) e soma a contribuição de cada resposta até
chegar à predição $f(x)$ da candidata.

### 6.3 Explicação global

In [ ]:
shap.plots.beeswarm(sv, max_display=15)

In [ ]:
shap.plots.bar(sv, max_display=15)

### 6.4 SHAP com lente de justiça: decompondo a diferença entre grupos

Como os valores SHAP são **aditivos**, a diferença de score médio entre dois grupos se decompõe exatamente:

$$\bar f_{\text{B/A}} - \bar f_{\text{PPI}} = \sum_i \big(\bar\phi_{i,\text{B/A}} - \bar\phi_{i,\text{PPI}}\big)$$

Cada termo diz **quanto da diferença entre os grupos passa por cada variável**.

In [ ]:
G_exp = G_te.loc[X_exp.index]
vals = pd.DataFrame(sv.values, columns=FEATURES, index=X_exp.index)
dif = (vals[G_exp["ppi"] == "Branca/Amarela"].mean() - vals[G_exp["ppi"] == "PPI"].mean())
dif = dif.reindex(dif.abs().sort_values().index)

total = (logodds(X_exp[G_exp["ppi"] == "Branca/Amarela"]).mean() - logodds(X_exp[G_exp["ppi"] == "PPI"]).mean())
print(f"Diferença de log-odds médio (Branca/Amarela − PPI): {total:.3f} | soma das contribuições: {dif.sum():.3f}")

top = dif.tail(12)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(top.index, top.values, color=[AZUL if v > 0 else LARANJA for v in top.values], height=0.6)
ax.axvline(0, color=TINTA2, linewidth=1); ax.grid(axis="y", visible=False)
ax.set_xlabel("Contribuição para a diferença de score (log-odds)\nazul: favorece Branca/Amarela · laranja: favorece PPI")
ax.set_title("Por onde passa a diferença entre os grupos"); plt.tight_layout(); plt.show()

### 6.5 Variáveis correlacionadas: qual é a "ausência" certa?

Renda, bens, escolaridade dos pais e escola andam juntos. O SHAP precisa decidir o que significa "não conhecer"
uma variável:

* **`tree_path_dependent`** (padrão): usa a estrutura das árvores, aproximadamente a distribuição *condicional*;
* **`interventional`**: sorteia valores de um *background* independentemente das outras variáveis. É uma
  intervenção, como no QII.

As duas respondem perguntas diferentes e podem ordenar as variáveis de forma diferente.

In [ ]:
sub = X_exp.iloc[:1000]
sv_int = shap.TreeExplainer(modelo, data=fundo.iloc[:100], feature_perturbation="interventional")(sub)
comp = pd.DataFrame({
    "path-dependent": np.abs(sv.values[:1000]).mean(0),
    "interventional": np.abs(sv_int.values).mean(0),
}, index=FEATURES)
comp["posição (path)"] = comp["path-dependent"].rank(ascending=False).astype(int)
comp["posição (interv.)"] = comp["interventional"].rank(ascending=False).astype(int)
comp.sort_values("path-dependent", ascending=False).head(12)

## 7. QII — Quantitative Input Influence

**Ideia (Datta, Sen & Zick, 2016):** a influência de uma variável é o quanto uma **quantidade de interesse**
(QoI) muda quando **intervimos** nela, substituindo seu valor por um valor sorteado da população e quebrando
sua ligação com as demais variáveis:

$$\iota(i) = Q(X) - \mathbb{E}\big[Q(X_{-i}\,U_i)\big]$$

A grande vantagem para fairness é que a QoI pode ser qualquer coisa:

| QoI | Pergunta respondida |
|---|---|
| decisão de um indivíduo | "o que mudou a decisão sobre esta candidata?" |
| taxa de seleção geral | "o que faz o modelo aprovar muita ou pouca gente?" |
| **disparidade** = seleção(B/A) − seleção(PPI) | "**quanto da desigualdade** se deve a cada variável?" |

### 7.1 Implementação

In [ ]:
N_QII = 30_000
X_q = X_te.sample(min(N_QII, len(X_te)), random_state=SEED)
g_q = G_te.loc[X_q.index, "ppi"].values
decide = lambda Z: (modelo.predict_proba(Z)[:, 1] >= 0.5).astype(int)

def qoi_disparidade(Z):
    d = decide(Z)
    return d[g_q == "Branca/Amarela"].mean() - d[g_q == "PPI"].mean()

def qoi_taxa(Z):
    return decide(Z).mean()

def intervir(Z, colunas, r):
    # Substitui o conjunto de colunas por valores de outra pessoa sorteada:
    # mantém a correlação DENTRO do conjunto e quebra a ligação com o resto.
    Z = Z.copy()
    doadores = r.permutation(len(Z))
    Z[colunas] = Z[colunas].values[doadores]
    return Z

def qii_unario(qoi, colunas_por_item, reps=5, seed=SEED):
    r = np.random.default_rng(seed)
    base = qoi(X_q)
    return pd.Series({nome: base - np.mean([qoi(intervir(X_q, cols, r)) for _ in range(reps)])
                      for nome, cols in colunas_por_item.items()})

print(f"Disparidade observada no modelo: {qoi_disparidade(X_q):.3f}  (pontos de taxa de seleção)")

### 7.2 QII unário: uma variável de cada vez

In [ ]:
qii_un = qii_unario(qoi_disparidade, {c: [c] for c in FEATURES}).sort_values()
top = qii_un.tail(12)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(top.index, top.values, color=AZUL, height=0.6); ax.grid(axis="y", visible=False)
ax.set_xlabel("Redução da disparidade ao intervir na variável (pontos de taxa de seleção)")
ax.set_title("QII unário sobre a disparidade PPI × Branca/Amarela"); plt.tight_layout(); plt.show()
print(f"Soma dos QII unários: {qii_un.sum():.3f}  vs. disparidade total: {qoi_disparidade(X_q):.3f}")

A soma dos QII unários raramente bate com o total. Com variáveis **redundantes** (renda, bens e computador
carregam informação parecida), intervir em uma só muda pouco, porque as outras "cobrem" a ausência.
Por isso o QII define a influência de **conjuntos** e a agrega com **valores de Shapley**.

### 7.3 QII de conjuntos + Shapley QII

Agrupamos as perguntas em blocos temáticos. Com 8 blocos há $2^8 = 256$ coalizões, então dá para calcular o
Shapley **exato**. Usamos os mesmos sorteios em todas as coalizões (*números aleatórios comuns*) para reduzir o ruído.

In [ ]:
BLOCOS = {
    "Renda": ["Renda familiar"],
    "Escolaridade dos pais": ["Escolaridade do pai", "Escolaridade da mãe"],
    "Ocupação dos pais": ["Ocupação do pai", "Ocupação da mãe"],
    "Moradia e bens": [NOMES[q] for q in ["Q005"] + [f"Q{i:03d}" for i in range(7, 24)]],
    "Acesso digital": ["Computador", "Internet"],
    "Tipo de escola": ["Escola privada"],
    "Território (UF)": ["UF da prova"],
    "Idade e língua": ["Faixa etária", "Prova de espanhol"],
}
r = np.random.default_rng(SEED)
perm = {b: r.permutation(len(X_q)) for b in BLOCOS}   # um sorteio fixo por bloco

def v_qii(S, qoi):
    Z = X_q.copy()
    for b in S:
        Z[BLOCOS[b]] = X_q[BLOCOS[b]].values[perm[b]]
    return qoi(X_q) - qoi(Z)        # influência do conjunto S

def shapley_qii(qoi):
    nomes = list(BLOCOS); n = len(nomes)
    cache = {frozenset(S): v_qii(S, qoi) for k in range(n + 1) for S in itertools.combinations(nomes, k)}
    phi = {}
    for i in nomes:
        outros = [j for j in nomes if j != i]
        phi[i] = sum(math.factorial(len(S)) * math.factorial(n - len(S) - 1) / math.factorial(n)
                     * (cache[frozenset(S) | {i}] - cache[frozenset(S)])
                     for k in range(n) for S in itertools.combinations(outros, k))
    return pd.Series(phi), cache[frozenset(nomes)]

phi_qii, total_qii = shapley_qii(qoi_disparidade)
phi_qii = phi_qii.sort_values()
print(f"Disparidade removida ao intervir em todos os blocos: {total_qii:.3f} | soma dos Shapley QII: {phi_qii.sum():.3f}")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(phi_qii.index, phi_qii.values, color=[AZUL if v > 0 else LARANJA for v in phi_qii.values], height=0.6)
ax.axvline(0, color=TINTA2, linewidth=1); ax.grid(axis="y", visible=False)
for i, v_ in enumerate(phi_qii.values):
    ax.text(v_ + (0.002 if v_ >= 0 else -0.002), i, f"{v_:.3f}", va="center",
            ha="left" if v_ >= 0 else "right", fontsize=9, color=TINTA2)
pad = 0.18 * (phi_qii.max() - min(phi_qii.min(), 0))
ax.set_xlim(min(phi_qii.min(), 0) - pad, phi_qii.max() + pad)
ax.set_xlabel("Contribuição para a disparidade de seleção (pontos)")
ax.set_title("Shapley QII: quanto da disparidade passa por cada bloco"); plt.tight_layout(); plt.show()

### 7.4 QII para a candidata

A mesma maquinaria explica uma decisão individual. A QoI é a probabilidade prevista para a candidata, e
a intervenção substitui um bloco por respostas de pessoas sorteadas.

In [ ]:
def qii_individuo(x, reps=300, seed=SEED):
    r = np.random.default_rng(seed)
    base = f_proba(x)[0]
    out = {}
    for b, cols in BLOCOS.items():
        Z = pd.concat([x] * reps, ignore_index=True)
        Z[cols] = X_q[cols].values[r.integers(0, len(X_q), reps)]
        out[b] = f_proba(Z).mean() - base   # quanto a probabilidade SOBE se "esquecermos" o bloco
    return pd.Series(out).sort_values()

qii_cand = qii_individuo(x_cand)
qii_cand.to_frame("Δ P(top 25%) ao intervir no bloco")

Valores positivos indicam que a resposta da candidata naquele bloco **puxa a probabilidade para baixo**:
trocá-la por uma resposta típica da população aumentaria sua chance.

## 8. Comparando os métodos

Colocamos lado a lado, por bloco, a parcela da diferença entre grupos atribuída pelo **SHAP** (em log-odds) e
pelo **Shapley QII** (em taxa de seleção). As unidades diferem, então comparamos as **proporções**.

In [ ]:
shap_bloco = pd.Series({b: dif[cols].sum() for b, cols in BLOCOS.items()})
tabela = pd.DataFrame({
    "SHAP (parcela da diferença de score)": shap_bloco / shap_bloco.sum(),
    "Shapley QII (parcela da disparidade de seleção)": phi_qii / phi_qii.sum(),
}).sort_values("Shapley QII (parcela da disparidade de seleção)")

fig, ax = plt.subplots(figsize=(10, 5))
y_ = np.arange(len(tabela)); h = 0.38
ax.barh(y_ + h / 2, tabela.iloc[:, 0], height=h * 0.92, color=AZUL, label="SHAP")
ax.barh(y_ - h / 2, tabela.iloc[:, 1], height=h * 0.92, color=LARANJA, label="Shapley QII")
ax.set_yticks(y_, tabela.index); ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1)); ax.axvline(0, color=TINTA2, linewidth=1)
ax.set_xlabel("Parcela da diferença entre Branca/Amarela e PPI"); ax.legend(loc="lower right")
ax.set_title("SHAP × QII: quais blocos explicam a desigualdade?"); plt.tight_layout(); plt.show()
tabela

| | LIME | SHAP | QII |
|---|---|---|---|
| Escopo | local | local e global (agregando) | local, global, **qualquer QoI** (inclusive disparidades) |
| Princípio | modelo linear substituto | valores de Shapley (atribuição aditiva) | intervenção causal nas entradas + Shapley |
| Garantias | nenhuma (heurística) | eficiência, simetria, dummy, aditividade | as mesmas do Shapley, sobre a QoI escolhida |
| Estabilidade | baixa (amostragem) | alta (TreeSHAP exato) | média (depende dos sorteios) |
| Correlação entre variáveis | ignora (perturbações irreais) | depende da escolha de "ausência" | quebra correlações de propósito, por isso usa conjuntos |
| Custo | médio | baixo para árvores | alto (muitas predições) |

## 9. Discussão

* **Explicar não é justificar.** Se o modelo rejeita por causa da renda ou da escola, isso descreve o mecanismo,
  mas não diz se o critério é legítimo. A desigualdade de oportunidades está *nos dados*.
* **Proxies.** Cor/raça não é entrada, mas a disparidade "passa" por renda, escola e bens. Quais desses caminhos
  seriam aceitáveis numa política de bolsas?
* **Explicações podem ser manipuladas.** Slack et al. (2020) mostram modelos discriminatórios desenhados para
  enganar o LIME e o SHAP, que apresentam explicações "inocentes".
* **Contestação.** Qual dos três métodos você daria à candidata da seção 4 para que ela entendesse e
  contestasse a decisão? E a um órgão regulador?

## 10. Exercícios

1. **Modelo "consciente".** Inclua cor/raça como entrada, re-treine e calcule o QII unário de cor/raça sobre
   a disparidade. Compare com a influência *indireta* que você mediu sem a variável.
2. **Estabilidade do LIME.** Varie `num_samples` (500, 2.000, 10.000) e `kernel_width`. Quando o top 5 se estabiliza?
3. **Outro grupo.** Repita a decomposição SHAP (6.4) e o Shapley QII (7.3) para escola pública × privada,
   retirando o bloco "Tipo de escola".
4. **Contrafactual.** Para a candidata, qual a *menor* mudança em um bloco que faria o modelo aprová-la?
   Compare com o que o SHAP e o QII sugeriam.
5. **Reflexão.** Em um parágrafo, defenda qual método deveria acompanhar decisões automatizadas sobre
   candidatos (pense no direito à explicação da LGPD, art. 20).

## Referências

* Ribeiro, Singh & Guestrin (2016). *"Why Should I Trust You?": Explaining the Predictions of Any Classifier.* KDD.
* Lundberg & Lee (2017). *A Unified Approach to Interpreting Model Predictions.* NeurIPS.
* Lundberg et al. (2020). *From local explanations to global understanding with explainable AI for trees.* Nature Machine Intelligence.
* Datta, Sen & Zick (2016). *Algorithmic Transparency via Quantitative Input Influence.* IEEE S&P.
* Slack et al. (2020). *Fooling LIME and SHAP: Adversarial Attacks on Post hoc Explanation Methods.* AIES.
* Molnar. *Interpretable Machine Learning* (christophm.github.io/interpretable-ml-book).
* INEP. *Microdados do ENEM 2023* e dicionário de dados.